# Análisis de ventas de farmacias: productos, categorías y más

**Objetivo:** sacar conclusiones del Excel completo (`Pharmacy_data.xlsx`): qué producto es el top de cada categoría, qué categorías son más rentables, cómo evolucionan las ventas y qué efecto tienen las promociones.

**Cómo usarlo:** ejecuta las celdas en orden. Tras cada gráfico hay una celda ✍️ para que escribas tu propia conclusión.

## Paso 0. Cargar y unir las hojas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap

plt.rcParams["figure.dpi"] = 100
pd.options.display.float_format = "{:,.2f}".format

RUTA = "../Data/Pharmacy_data.xlsx"   # ajusta la ruta si hace falta

Datos = pd.read_excel(RUTA, sheet_name=None)
fact      = Datos["FactSales"]
farmacia  = Datos["DimPharmacy"]
producto  = Datos["DimProduct"]

data = (
    fact
    .merge(farmacia[["PharmacyID", "Country"]], on="PharmacyID", how="left", validate="many_to_one")
    .merge(producto[["ProductID", "ProductName", "Category", "Brand", "IsGeneric"]],
           on="ProductID", how="left", validate="many_to_one")
)
data["Fecha"] = pd.to_datetime(data["DateKey"].astype(str), format="%Y%m%d")
data["Mes"] = data["Fecha"].dt.to_period("M").dt.to_timestamp()

print(data.shape, "| nulos:", data.isna().sum().sum())
data.head()

### Ojo: hay nombres de producto repetidos

Hay 220 productos (`ProductID`) pero menos nombres distintos. Si agrupas solo por `ProductName`, mezclas productos diferentes. Por eso agrupamos siempre por **`ProductID`** y usamos el nombre solo como etiqueta.

In [ ]:
print("Productos (ID):", producto["ProductID"].nunique())
print("Nombres distintos:", producto["ProductName"].nunique())

repetidos = (producto[producto["ProductName"].duplicated(keep=False)]
             .sort_values("ProductName")[["ProductID", "ProductName", "Category", "PackSize"]])
repetidos.head(10)

## Paso 1. Cifras generales

In [ ]:
ing, cost, mar = data["RevenueEUR"].sum(), data["CostEUR"].sum(), data["MarginEUR"].sum()

kpis = pd.Series({
    "Ventas (filas)": len(data),
    "Unidades": data["UnitsSold"].sum(),
    "Ingresos (€)": ing,
    "Costes (€)": cost,
    "Margen (€)": mar,
    "Margen (%)": mar / ing * 100,
})
kpis

## Paso 2. Producto top de cada categoría

Depende del criterio: el producto que más **ingresa** no tiene por qué ser el que más **unidades** vende ni el que más **margen** deja.

In [ ]:
prod = (data.groupby(["Category", "ProductID", "ProductName"], as_index=False)
        .agg(ventas=("SalesID", "count"),
             unidades=("UnitsSold", "sum"),
             ingresos=("RevenueEUR", "sum"),
             margen=("MarginEUR", "sum")))
prod["margen_pct"] = prod["margen"] / prod["ingresos"] * 100
prod["Producto"] = prod["ProductName"] + " (" + prod["ProductID"] + ")"

def top_por(criterio):
    t = prod.sort_values(criterio, ascending=False).groupby("Category").head(1)
    return t.sort_values("Category").set_index("Category")[["Producto", criterio]]

pd.concat({"Por ingresos": top_por("ingresos"),
           "Por unidades": top_por("unidades"),
           "Por margen (€)": top_por("margen")}, axis=1)

In [ ]:
# Top 3 por ingresos en cada categoría
(prod.sort_values("ingresos", ascending=False)
     .groupby("Category").head(3)
     .sort_values(["Category", "ingresos"], ascending=[True, False])
     [["Category", "Producto", "unidades", "ingresos", "margen", "margen_pct"]]
     .reset_index(drop=True))

In [ ]:
# Top 5 por ingresos en cada categoría
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for ax, (cat, g) in zip(axes, prod.groupby("Category")):
    t = g.nlargest(5, "ingresos").sort_values("ingresos")
    etiquetas = [textwrap.fill(n, 30) for n in t["Producto"]]
    ax.barh(etiquetas, t["ingresos"])
    ax.set_title(cat, fontweight="bold")
    ax.set_xlabel("Ingresos (miles de €)")
    ax.xaxis.set_major_formatter(lambda x, p: f"{x/1000:.0f}k")
    ax.tick_params(axis="y", labelsize=8)

for ax in axes[len(prod["Category"].unique()):]:
    ax.axis("off")

fig.suptitle("Top 5 productos por ingresos en cada categoría", fontsize=14)
plt.tight_layout()
plt.show()

✍️ **Tu conclusión (producto top por categoría):**

- ¿Coincide el top por ingresos con el top por unidades? ¿En qué categorías no?
- ...

## Paso 3. Top 10 productos de toda la tienda

In [ ]:
top10 = prod.nlargest(10, "ingresos").sort_values("ingresos")

plt.figure(figsize=(10, 6))
plt.barh([textwrap.fill(n, 40) for n in top10["Producto"]], top10["ingresos"])
plt.xlabel("Ingresos (€)")
plt.title("Top 10 productos por ingresos")
plt.tight_layout()
plt.show()

print("Categorías de esos 10 productos:")
print(top10["Category"].value_counts())
print("Peso del top 10 sobre el total:", round(top10["ingresos"].sum() / data["RevenueEUR"].sum() * 100, 1), "%")

✍️ **Tu conclusión (top 10):** ¿Qué categoría domina? ¿Las ventas están concentradas en pocos productos o repartidas?

...

## Paso 4. Categorías: ¿cuál vende más y cuál es más rentable?

Vender mucho no es lo mismo que ser rentable. Comparamos ingresos y **margen %** (margen / ingresos).

In [ ]:
cat = (data.groupby("Category")
       .agg(unidades=("UnitsSold", "sum"),
            ingresos=("RevenueEUR", "sum"),
            margen=("MarginEUR", "sum")))
cat["margen_pct"] = cat["margen"] / cat["ingresos"] * 100
cat["peso_ingresos_%"] = cat["ingresos"] / cat["ingresos"].sum() * 100
cat = cat.sort_values("ingresos", ascending=False)
cat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cat.index, cat["ingresos"])
axes[0].set_title("Ingresos por categoría (€)")
axes[0].tick_params(axis="x", rotation=30)

barras = axes[1].bar(cat.index, cat["margen_pct"], color="tab:green")
axes[1].set_title("Margen (%) por categoría")
axes[1].tick_params(axis="x", rotation=30)
axes[1].bar_label(barras, fmt="%.1f%%")

plt.tight_layout()
plt.show()

✍️ **Tu conclusión (categorías):** ¿Qué categoría ingresa más? ¿Es también la que más margen % deja?

...

## Paso 5. Evolución mensual por categoría

In [ ]:
mes_cat = (data.groupby(["Mes", "Category"])["RevenueEUR"].sum().unstack())

mes_cat.plot(figsize=(12, 5), marker="o", markersize=3)
plt.title("Ingresos mensuales por categoría")
plt.ylabel("Ingresos (€)")
plt.xlabel("")
plt.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Ingresos totales por mes
total_mes = data.groupby("Mes")["RevenueEUR"].sum()

plt.figure(figsize=(12, 4))
plt.plot(total_mes.index, total_mes.values, marker="o")
plt.title("Ingresos totales por mes")
plt.ylabel("Ingresos (€)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Mejor mes:", total_mes.idxmax().strftime("%Y-%m"), "|", round(total_mes.max(), 2), "€")
print("Peor mes :", total_mes.idxmin().strftime("%Y-%m"), "|", round(total_mes.min(), 2), "€")

✍️ **Tu conclusión (evolución):** ¿Hay estacionalidad o tendencia? ¿Alguna categoría se comporta distinto?

...

## Paso 6. Efecto de las promociones

Compara el margen % de las ventas con promoción (`PromoFlag = Yes`) y sin ella.

In [ ]:
promo = (data.groupby(["Category", "PromoFlag"])
         .agg(ingresos=("RevenueEUR", "sum"), margen=("MarginEUR", "sum")))
promo["margen_pct"] = promo["margen"] / promo["ingresos"] * 100
tabla_promo = promo["margen_pct"].unstack()
tabla_promo["diferencia (pp)"] = tabla_promo["Yes"] - tabla_promo["No"]
tabla_promo

In [ ]:
tabla_promo[["No", "Yes"]].plot(kind="bar", figsize=(10, 5))
plt.title("Margen (%) con y sin promoción, por categoría")
plt.ylabel("Margen (%)")
plt.xlabel("")
plt.xticks(rotation=30)
plt.legend(title="Promoción")
plt.tight_layout()
plt.show()

peso = data.groupby("PromoFlag")["SalesID"].count()
print("Porcentaje de ventas con promoción:", round(peso["Yes"] / peso.sum() * 100, 1), "%")

✍️ **Tu conclusión (promociones):** ¿Cuánto margen se pierde al promocionar? ¿Compensa vender más unidades?

...

## Paso 7. Países

In [ ]:
pais = (data.groupby("Country")
        .agg(ingresos=("RevenueEUR", "sum"), margen=("MarginEUR", "sum"), unidades=("UnitsSold", "sum")))
pais["margen_pct"] = pais["margen"] / pais["ingresos"] * 100
pais = pais.sort_values("ingresos")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(pais.index, pais["ingresos"])
axes[0].set_title("Ingresos por país (€)")
axes[1].barh(pais.index, pais["margen_pct"], color="tab:green")
axes[1].set_title("Margen (%) por país")
plt.tight_layout()
plt.show()

In [ ]:
# Producto top (por ingresos) de cada país
pp = (data.groupby(["Country", "ProductID", "ProductName"], as_index=False)["RevenueEUR"].sum())
pp["Producto"] = pp["ProductName"] + " (" + pp["ProductID"] + ")"
(pp.sort_values("RevenueEUR", ascending=False).groupby("Country").head(1)
   .sort_values("Country").set_index("Country")[["Producto", "RevenueEUR"]])

✍️ **Tu conclusión (países):** ¿Qué país ingresa más? ¿Hay diferencias de margen entre países?

...

## Paso 8. Marcas y genéricos

In [ ]:
marca = (data.groupby("Brand")
         .agg(ingresos=("RevenueEUR", "sum"), margen=("MarginEUR", "sum"))
         .sort_values("ingresos", ascending=False))
marca["margen_pct"] = marca["margen"] / marca["ingresos"] * 100
print("Top 5 marcas por ingresos:")
display(marca.head(5))

gen = (data.groupby("IsGeneric")
       .agg(ingresos=("RevenueEUR", "sum"), margen=("MarginEUR", "sum"), unidades=("UnitsSold", "sum")))
gen["margen_pct"] = gen["margen"] / gen["ingresos"] * 100
gen["ingreso_por_unidad"] = gen["ingresos"] / gen["unidades"]
gen

## Paso 9. Conclusiones finales

Resume en 5-6 líneas lo más importante que has encontrado: producto top por categoría, categoría más rentable, efecto de las promociones y cualquier sorpresa.

✍️ **Tus conclusiones:**

...